# **Aplicação Web para Previsão de Séries Temporais com Chronos-2**

Este notebook documenta o desenvolvimento de uma [aplicação interativa](https://chronoslens-38662419748.us-west1.run.app/) projetada para realização de previsões avançadas de séries temporais de maneira facilitada. A ideia central é estruturar um fluxo simplificado onde qualquer pessoa deve conseguir fazer upload dos seus dados, analisá-los e realizar previsões de alta qualidade com suporte a variáveis exógenas usando o modelo de fundação [Chronos-2](https://huggingface.co/amazon/chronos-2), da Amazon Science. [Acesse o app aqui.](https://chronoslens-38662419748.us-west1.run.app/)

![ChronosLens — Landing Page](assets/home.png)

## **Proposta da Aplicação**

A previsão de séries temporais é um processo trabalhoso e que requer vasto conhecimento de pressupostos,  modelos estatísticos, ou desenvolvimento de redes neurais complexas, além de técnicas de treinamento e validação adaptados para cada novo conjunto de dados. Recentemente, a **Amazon Science** publicou o **Chronos-2**, um Foundation Model treinado em bilhões de pontos temporais que trata séries como linguagem e realiza previsões **zero-shot** (sem necessidade de treinamento prévio nos dados do usuário).

A ideia foi envelopar essa tecnologia em uma interface para facilitar o uso, onde o fluxo completo, desde o carregamento dos dados até a projeção final fosse executado em um único pipeline guiado, sem expor qualquer complexidade técnica para o usuário final.

## **Tecnologias**

As tecnologias utilizadas foram escolhidas com foco no fácil e rápido desenvolvimento, interatividade e estética:

| Camada | Tecnologia |
|---|---|
| Linguagem | Python 3.11 |
| Interface | Streamlit |
| Visualização | Plotly |
| Modelo Preditivo | Chronos-2 (HuggingFace) |
| Gerenciamento de Pacotes | `uv` |
| Deploy | Docker + Google Cloud Run |


## **Funcionalidades**

O pipeline da aplicação é dividido em quatro etapas sequenciais, cada uma desbloqueada após a conclusão da anterior.

### **A. Carregar Dados**
Aceita arquivos CSV, Excel e Parquet. O app detecta automaticamente a coluna de data/hora e identifica lacunas (dados ausentes) na série. Para usuários que queiram apenas explorar, há conjuntos de demonstração embutidos.
![ChronosLens — Tela de Carregar Dados com dados de demonstração carregados](assets/carregamento_dados.png)

### **B. Análise Exploratória de Dados (EDA)**
O aplicativo gera três análises principais: (1) Visualização da série histórica e distribuição, (2) Decomposição Sazonal (Tendência + Sazonalidade + Resíduo) e (3) Correlações e ACF/PACF. A Análise Exploratória é uma etapa essencial para entender os dados antes de gerar qualquer previsão e fundamentar análise posterior dos resultados obtidos.
![ChronosLens — Análise Exploratória: métricas descritivas e série temporal interativa](assets/eda.png)

### **C. Backtesting**
A etapa de backtesting permite avaliar o modelo em dados de teste (com ou sem validação cruzada), simulando como o Chronos-2 teria se comportado em dados já conhecidos, garantindo que o modelo se adequa ao problema antes de projetar para o futuro.
![ChronosLens — Backtesting com Treino e Teste Simples ou Validação Cruzada](assets/backtesting.png)

### **D. Previsão com Suporte Multivariado**
O modelo dá suporte a variáveis exógenas com valores futuros conhecidos (feriados, temperatura prevista, orçamento de marketing) podem ser informadas para enriquecer as projeções, como apenas conhecidas no passado.
![ChronosLens — Previsão com suporte a variáveis exógenas e Intervalo de Confiança.](assets/previsao.png)

## **Deploy com Google Cloud Run**

Para tornar a aplicação acessível de forma contínua, o projeto foi containerizado e publicado no **Google Cloud Run** via integração nativa com o GitHub, de modo que qualquer commit na branch `main` aciona automaticamente um novo build e implantação (CI/CD).

O `Dockerfile` utiliza a imagem Python Slim e utiliza `uv` para gerenciamento de dependências, que resolve e instala o ambiente com precisão e velocidade superiores ao `pip`.

In [ ]:
FROM python:3.11-slim
ENV PORT=8080
COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/uv
WORKDIR /app
COPY pyproject.toml uv.lock ./
COPY . .
RUN uv sync --frozen --no-dev
EXPOSE 8080
CMD ["streamlit", "run", "main.py", "--server.port", "8080", "--server.address", "0.0.0.0"]

## **Referências**
- [Ansari, A.F. et al. (2024). *Chronos: Learning the Language of Time Series*. arXiv:2403.07815.](https://arxiv.org/abs/2403.07815)
- [Repositório oficial: amazon-science/chronos-forecasting](https://github.com/amazon-science/chronos-forecasting)
- [Streamlit: A faster way to build and share data apps](https://docs.streamlit.io/)